# Review Notebook: Rule-Based Sentiment Analysis

**For studying, not presenting.** It explains everything in `RuleBased_Presentation.ipynb`: how VADER calculates a score, how we built our bigger dictionary, the bug we found and fixed, and why rules stop at about 88–89%.

| Part | What's inside |
|---|---|
| 1 | Results and the story in one table |
| 2 | Data decisions |
| 3 | How VADER works, by hand |
| 4 | Our improvements, step by step (with demos) |
| 5 | Why rules hit a ceiling, and is it still "rule-based"? |
| 6 | Lesson connections |
| 7 | Questions the professor might ask |
| 8 | Self-check quiz |

**How to study:** run a cell, change a word or a number, and predict the result before running it again. For Parts 7–8, answer out loud first, then click ▶.

Runs on Colab or locally (kernel **Python (sentiment_env)**). Run `RuleBased_Presentation.ipynb` first, since some cells load `model/rule_based.json`.

In [1]:
# only runs on Google Colab: connect Google Drive and go to the project folder
import os, sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/Sentiment Analysis")   # change this if your folder has a different name
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vaderSentiment"])

import re
import json
import numpy as np
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer, NEGATE, BOOSTER_DICT
vader = SentimentIntensityAnalyzer()

---
# Part 1: Results

| Step | What we added | Validation | Test |
|---|---|---|---|
| 1 | Plain VADER (default cut-off 0.05) | 79.3% | 79.6% |
| 2 | + 27 hand-picked restaurant words, "never again" rule, tuned cut-off | 83.4% | 83.2% |
| 3 | + ~1,040 words scored from training reviews (+44 negation words scored separately), raw score ÷ √length | 87.8% | — |
| 4 | + 1,258 two-word and 740 three-word phrase rules | 88.8% | — |
| 5 | + 7 pattern rules | 89.0% | — |
| 6 | + star-rating and question-mark rules, 1.5× weight for negation words and patterns (final) | **89.4%** | **88.6%** |

(Only the three versions we present were scored on the test set.)

**The story in one sentence:** *Plain VADER doesn't know restaurant language, so we taught it: first by hand, then with a dictionary scored from the training reviews, then with phrases and patterns. That took it from 80% to almost 89%, and the remaining mistakes need an understanding of context that word lists don't have.*

---
# Part 2: Data decisions

| Decision | Why |
|---|---|
| `read_csv(..., encoding="latin-1")` | The file isn't valid UTF-8, so the default read crashes |
| Repair `ï¿½` junk (→ apostrophe between letters, else space) | Especially important for rules: "didnï¿½t" must become "didn't" so VADER sees a negation |
| **No** `dropna()` | Only latitude/longitude are missing (660 rows, one store), and we don't use them |
| 1–2★ = negative, 4–5★ = positive, **3★ left out** | 3★ reviews are mixed, with no clear right answer. Amazon Review Polarity is built the same way |
| One copy per review text (majority label) | ~5,000 Google placeholder texts ("Excellent" ×2,166). Otherwise the same text is in both the data we build rules from and the data we test on |
| Split 70/15/15, stratified | **Train** = find words/phrases, **validation** = choose cut-off and settings, **test** = final score, once |

**"Rules don't train, so why split?"** Because we *built* the rules from data (Steps 3–4) and *chose* settings by checking accuracy (the cut-off). Measuring on the same reviews we built from would look better than it really is, the same overfitting idea as in Week 4.

---
# Part 3: How VADER works, by hand

## 3.1 The dictionary

In [2]:
print("words in VADER's dictionary:", len(vader.lexicon))
for w in ["good", "great", "excellent", "bad", "terrible", "rude", "haha", "slow", "cold", "waited"]:
    print(f"{w:10s} {vader.lexicon.get(w, 'not in dictionary')}")

words in VADER's dictionary: 7506
good       1.9
great      3.1
excellent  2.7
bad        -2.5
terrible   -2.1
rude       -2.0
haha       2.0
slow       not in dictionary
cold       not in dictionary
waited     not in dictionary


Every word was scored −4 to +4 by **human raters** (VADER's authors averaged ten people's ratings). That's the core idea of rule-based: the knowledge is put in by people, not learned by a model.

## 3.2 The compound score

VADER adds up the word scores (after applying its rules), then squashes the sum into −1…+1:

`compound = sum / √(sum² + 15)`

In [3]:
squash = lambda total: total / np.sqrt(total ** 2 + 15)
print(f"'The food was good': good = 1.9 -> {squash(1.9):.3f}   VADER: {vader.polarity_scores('The food was good')['compound']:.3f}")
print(f"'great':             great = 3.1 -> {squash(3.1):.3f}   VADER: {vader.polarity_scores('great')['compound']:.3f}")
for total in [2, 5, 10, 20, 40]:
    print(f"sum {total:3d} -> compound {squash(total):.3f}")

'The food was good': good = 1.9 -> 0.440   VADER: 0.440
'great':             great = 3.1 -> 0.625   VADER: 0.625
sum   2 -> compound 0.459
sum   5 -> compound 0.791
sum  10 -> compound 0.933
sum  20 -> compound 0.982
sum  40 -> compound 0.995


The squash works like an **activation function** (Week 1): big sums get close to ±1 but never pass it, a lot like tanh. Notice how sums of 10, 20 and 40 all come out at about 0.93–0.99. That's why we stopped using the compound score in Step 3 (see 4.3).

## 3.3 The rules in action

In [4]:
tests = [
    ("base",        "The food was good"),
    ("negation",    "The food was not good"),
    ("booster",     "The food was very good"),
    ("dampener",    "The food was kind of good"),
    ("CAPS",        "The food was GOOD"),
    ("exclamation", "The food was good!!!"),
    ("but-rule",    "The food was good but the service was slow"),
    ("but-rule 2",  "The service was slow but the food was good"),
]
for rule, s in tests:
    print(f"{rule:12s} {vader.polarity_scores(s)['compound']:+.3f}   {s}")

base         +0.440   The food was good
negation     -0.341   The food was not good
booster      +0.493   The food was very good
dampener     +0.383   The food was kind of good
CAPS         +0.562   The food was GOOD
exclamation  +0.583   The food was good!!!
but-rule     +0.238   The food was good but the service was slow
but-rule 2   +0.593   The service was slow but the food was good


| Rule | What it does |
|---|---|
| Negation | "not", "never", "isn't"… within 3 words before a sentiment word **flip it and weaken it** (× −0.74) |
| Booster / dampener | "very", "extremely" add strength; "kind of", "slightly" reduce it |
| CAPS | a word in ALL CAPS (when the rest isn't) gets stronger |
| Exclamation | each "!" (up to 4) adds a bit of intensity |
| "but" | words **before** "but" count half, words **after** count 1.5× |

The two "but" sentences have the same words but different scores. Whatever comes after "but" wins. That's a hand-written rule about English.

**Check:** "not good" = 1.9 × −0.74 = −1.41 → squash(−1.41) = ?

In [5]:
print(f"{squash(1.9 * -0.74):.3f}   VADER: {vader.polarity_scores('The food was not good')['compound']:.3f}")

-0.341   VADER: -0.341


---
# Part 4: Our improvements

## 4.1 Hand-picked words (Step 2)

In [6]:
hand = SentimentIntensityAnalyzer()
hand.lexicon.update({"waited": -1.5, "minutes": -1.0, "cold": -1.5, "slow": -1.5, "fast": 1.5, "hot": 1.0, "tasty": 2.0})
for s in ["I waited 40 minutes for cold fries", "Fast service and the fries were hot",
          "The fries were not cold", "Hot coffee spilled on me"]:
    print(f"plain {vader.polarity_scores(s)['compound']:+.3f}   ours {hand.polarity_scores(s)['compound']:+.3f}   {s}")

plain +0.000   ours -0.718   I waited 40 minutes for cold fries
plain +0.000   ours +0.542   Fast service and the fries were hot
plain +0.000   ours +0.276   The fries were not cold
plain +0.000   ours +0.250   Hot coffee spilled on me


- New words automatically work with VADER's rules: "not cold" is flipped.
- **Limit of any dictionary:** "Hot coffee spilled on me" gets a *positive* push from "hot". Each word has only one score, no matter the context.
- We can't hand-score thousands of words, which leads to Step 3.

## 4.2 Scoring words from the training reviews (Step 3)

`score = 0.5 × log( share of positive reviews containing the word / share of negative reviews containing the word )`

Worked example: a word appears in 300 of 6,575 negative reviews and 50 of 6,739 positive ones.

In [7]:
pos_with, pos_total = 50, 6739
neg_with, neg_total = 300, 6575
ratio = (pos_with / pos_total) / (neg_with / neg_total)
print(f"share in positive {pos_with / pos_total:.3%}, share in negative {neg_with / neg_total:.3%}, ratio {ratio:.3f}")
print(f"log ratio {np.log(ratio):.3f} -> score 0.5 x log = {0.5 * np.log(ratio):+.2f}")

share in positive 0.742%, share in negative 4.563%, ratio 0.163
log ratio -1.816 -> score 0.5 x log = -0.91


- **Why log?** A word 3× more common in negative reviews and one 3× more common in positive reviews should get scores of the same size with opposite signs. Log does that: log(3) = +1.10, log(1/3) = −1.10.
- **Why ×0.5?** It brings the scores to a size similar to VADER's own words. We tried 0.25, 0.5, 0.7, 1 and 2 on validation, and 0.5 was best.
- **Filters:** at least 5 training reviews (so 1–2 lucky reviews don't create a word), and |log ratio| ≥ 1, i.e. at least 2.7× more common in one class (only clear words).
- **The +1 in the code** ("add-one smoothing") avoids dividing by zero for words that never appear in one class.
- VADER's own 7,500 words keep their original scores.

In [8]:
saved = json.load(open("model/rule_based.json"))
words = pd.Series(saved["corpus_words"]).sort_values()
print("dictionary words we added:", len(words))
print("most negative:", ", ".join(f"{w} {s:+.1f}" for w, s in words.head(12).items()))
print("most positive:", ", ".join(f"{w} {s:+.1f}" for w, s in words.tail(12)[::-1].items()))
print("odd ones:", {w: round(saved["corpus_words"][w], 2) for w in ["portuguese", "rice", "oven", "gloves"] if w in saved["corpus_words"]})

dictionary words we added: 1042
most negative: disrespectful -2.0, answer -1.7, filthy -1.7, answered -1.7, raw -1.7, slowest -1.7, yelled -1.6, response -1.6, acknowledge -1.6, zero -1.5, poorly -1.5, pulled -1.5
most positive: speedy +1.5, affordable +1.4, prompt +1.3, oven +1.3, yum +1.3, attentive +1.2, customize +1.2, variety +1.1, digital +1.1, tasty +1.1, breakfasts +1.1, portuguese +1.1
odd ones: {'portuguese': 1.09, 'rice': 1.09, 'oven': 1.31, 'gloves': -1.48}


The odd ones ("portuguese", "rice", "oven") lean one way only because certain well-reviewed stores sell those items. A data-built dictionary picks up **any** word that differs between the classes, sentiment or not. We kept the formula's output as is instead of hand-removing words, and we say so openly.

## 4.3 Why we read the raw total instead of the compound score

In [9]:
big = SentimentIntensityAnalyzer()
big.lexicon.update(saved["corpus_words"])
sample = pd.read_csv("mcdo_reviews_clean.csv")["review"].sample(1000, random_state=0)
for name, analyzer in [("plain VADER", vader), ("with our 1,042 words", big)]:
    c = np.array([analyzer.polarity_scores(t)["compound"] for t in sample])
    print(f"{name:22s} share of reviews with |compound| > 0.9: {(np.abs(c) > 0.9).mean():.1%}")

plain VADER            share of reviews with |compound| > 0.9: 5.1%


with our 1,042 words   share of reviews with |compound| > 0.9: 16.3%


With 1,000+ extra scored words, reviews pile up bigger sums, and about three times as many get squashed close to ±1, where differences in strength disappear. In our first try with the compound score, accuracy actually *dropped* to 76–82%. With the raw total it went up to 87%. So we take VADER's total **before** squashing and divide by √(number of words), so long reviews don't win just by having more words.

## 4.4 The negation bug we found (good to know!)

In [10]:
broken = SentimentIntensityAnalyzer()
broken.lexicon["not"] = -0.64          # what happened when "not" was added to the dictionary
print("normal VADER :", vader.polarity_scores("The food was not good")["compound"])
print("'not' added  :", broken.polarity_scores("The food was not good")["compound"])
print("\nVADER's negation words:", sorted(NEGATE)[:15], "...")
print("negation/intensifier words we score separately:", {w: round(s, 2) for w, s in list(saved["negation_words"].items())[:10]})

normal VADER : -0.3412
'not' added  : 0.3094

VADER's negation words: ["ain't", 'aint', "aren't", 'arent', "can't", 'cannot', 'cant', "couldn't", 'couldnt', "daren't", 'darent', 'despite', "didn't", 'didnt', "doesn't"] ...
negation/intensifier words we score separately: {'extremely': -0.67, 'nope': -0.95, 'dont': -1.07, 'none': -0.97, "won't": -0.78, "wasn't": -0.63, 'total': -0.97, 'without': -0.52, 'almost': -0.59, 'occasional': 0.61}


VADER only treats "not" as a negation **if "not" is not in its dictionary**. Our formula gave "not", "didn't", "wasn't" and others small negative scores (complaints use them a lot), which quietly **switched off the negation rule**: "not good" became *positive*.

**The fix:** keep VADER's negation words (and intensifiers like "very") out of the dictionary, and add their points separately. Result: VADER's rule works again, and the information stays in. Validation went from 88.7% (with the bug) to 89.0%, so the correct version is also the better one.

**Lesson:** test your rules on small examples you know the answer to. We found this by checking "not bad at all".

## 4.5 Phrases (Step 4)

In [11]:
p2 = pd.Series(saved["phrases_2"]).sort_values()
print("two-word phrases:", len(p2))
for w in ["not fresh", "not friendly", "not bad", "go wrong", "worst ever", "bit slow", "little slow", "my go"]:
    if w in saved["phrases_2"]:
        print(f"  {w:14s} {1.5 * saved['phrases_2'][w]:+.2f}  (x1.5 as used in the score)")

two-word phrases: 1258
  not fresh      -2.23  (x1.5 as used in the score)
  not friendly   -2.18  (x1.5 as used in the score)
  not bad        +3.33  (x1.5 as used in the score)
  go wrong       +3.49  (x1.5 as used in the score)
  worst ever     +2.76  (x1.5 as used in the score)
  bit slow       +2.94  (x1.5 as used in the score)
  little slow    +3.04  (x1.5 as used in the score)
  my go          +2.98  (x1.5 as used in the score)


A phrase only gets a score for the **extra** meaning beyond its two words: `extra = log-odds(phrase) − log-odds(word 1) − log-odds(word 2)`. So a phrase score is a **correction**, and it can go either way:

- "not fresh" / "not friendly": extra **negative**, because "fresh"/"friendly" alone count as positive.
- "not bad" / "go wrong" (from "can't go wrong"): extra **positive**, because "bad"/"wrong" alone count as negative.
- **"worst ever" is positive (+2.76)!** Not because it's nice: "worst" and "ever" are each already strongly negative in our scores, and adding both would punish the review twice. The correction pulls it back. *Be ready for this question: it looks like a bug but it's the formula working as designed.*
- "bit slow" / "little slow": positive corrections, because people write "a bit slow but the food was great" in good reviews, so it's much milder than "slow" alone.
- "my go" comes from "my go-to McDonald's".

## 4.6 Patterns (Step 5)

Regular expressions catch things a word list can't, like a number followed by "minutes" or the typo "to long".

In [12]:
for name, rx, points in saved["patterns"]:
    print(f"{points:+.1f}  {name}")
test = ["Took 35 mins to get a coffee", "Service could of been better", "The line was to long",
        "Better than I expected!", "No complaints at all"]
for t in test:
    hits = [name for name, rx, _ in saved["patterns"] if re.search(rx, t, re.IGNORECASE)]
    print(f"{t:35s} -> {hits}")

-1.5  a number of minutes/hours (waiting time)
-2.0  could have been better / needs improvement
-2.0  the only good thing
-1.5  too long / too slow / too expensive
-2.0  never again / never coming back
+1.5  better than expected
+2.0  no complaints / can't complain
Took 35 mins to get a coffee        -> ['a number of minutes/hours (waiting time)']
Service could of been better        -> ['could have been better / needs improvement']
The line was to long                -> ['too long / too slow / too expensive']
Better than I expected!             -> ['better than expected']
No complaints at all                -> ["no complaints / can't complain"]


## 4.7 Star-rating and question rules (Step 6)

In [13]:
import re
stars = re.compile(saved["star_rules"]["stars"], re.IGNORECASE)
out_of_10 = re.compile(saved["star_rules"]["out_of_10"])
for t in ["Only giving 2 stars because the McMuffin was good", "5 stars, best breakfast in town", "Food was a 3/10", "zero stars if I could",
          "Why does it look like someone spit on my food?", "Who doesn't love McDonald's?"]:
    print(f"stars: {stars.search(t).group(0) if stars.search(t) else '-':8s} out of 10: {out_of_10.search(t).group(0) if out_of_10.search(t) else '-':5s} question: {'?' in t!s:5s}  {t}")

stars: 2 stars  out of 10: -     question: False  Only giving 2 stars because the McMuffin was good
stars: 5 stars  out of 10: -     question: False  5 stars, best breakfast in town
stars: -        out of 10: 3/10  question: False  Food was a 3/10
stars: zero stars out of 10: -     question: False  zero stars if I could
stars: -        out of 10: -     question: True   Why does it look like someone spit on my food?
stars: -        out of 10: -     question: True   Who doesn't love McDonald's?


- **Star rule:** some people write their rating in the text. 1–2 stars (or 0–4/10) → −3 points; 4–5 stars (or 7–10/10) → +3. On validation it fired on 61 reviews and pointed the right way in 95% of them.
- **Question rule:** a "?" → −1 point. 80% of validation reviews with a question were complaints ("Why does it look like someone spit on my food?").
- Both are **simple, explainable rules**, but neither is perfect: "Only giving 2 stars because the McMuffin was good" is *correctly* negative, and "Who doesn't love McDonald's?" gets a wrong push from the question rule (and from negation flipping "love").
- **1.5× weights:** on validation, negation-word points and pattern points worked slightly better counted 1.5×. We only tried 1 and 1.5 for each, to avoid fine-tuning on validation.

## 4.8 The cut-off

The final score is positive or negative depending on whether it's above the cut-off (**−0.254**), which we chose on the validation set by trying 121 values. It's slightly below 0 because the added words and phrases lean a bit negative overall (complaints use more varied vocabulary), so a review with a total of exactly 0 is more likely positive.

---
# Part 5: The ceiling, and is it still rule-based?

## Why we stopped at ~89%

| More things we tried (validation) | Result |
|---|---|
| Bigger dictionary (lower filters) | worse (more noisy words) |
| Stronger word scores (×0.7, ×1, ×2) | worse |
| Changing VADER's own words where reviews disagree ("respect", "help", "care" lean negative in reviews) | slightly worse |
| Separate weights for each part | +0.1 point at most |
| Keeping rare words with shrunken scores instead of a hard filter | worse |
| Giving the last sentence more weight | worse |
| A rule for prices ("$30 for…") | no change |
| **Star-rating + question rules, 1.5× weights** | **+0.4 points, kept** |

Each idea changed only a handful of reviews, and nothing got past ~89.5%. Tuning more would just fit the validation set.

**The bigger reason:** in earlier experiments, machine-learning models that look at which words appear (logistic regression, SVM, Naive Bayes, a small neural network) **all stopped at about 90%** on this data. A rule-based system works from the same word information, so it can't really go past that. The remaining mistakes are:

1. **context:** "They were fast, but my fries were cold", "I asked for no pickles and got extra pickles"
2. **sarcasm:** "Oh great, cold fries again"
3. **stars that contradict the text:** "Great place Moderate Prices" with 1–2 stars

## Is it still rule-based if we used data to build the dictionary?

**Yes.** This is called a **corpus-based lexicon**, a standard way to build domain dictionaries. The final system is a fixed, readable list of words, phrases and patterns, applied with fixed rules. Nothing is trained during scoring, and you can open `model/rule_based.json` and see every number.

It's honest to admit the line is blurry, though: scoring words by their log-odds is closely related to how the **Naive Bayes** classifier works. The difference is that our scores plug into VADER's hand-written rules (negation, "but", intensifiers, caps), and our phrase and pattern rules are designed by us.

## Pros and cons

| Pros | Cons |
|---|---|
| No GPU, instant, works anywhere | Only as good as its word lists |
| Fully explainable: every point is traceable | One score per word, no context ("hot fries" vs "hot coffee spilled") |
| Easy to adjust for a domain | Typos, slang, sarcasm break it |
| Doesn't need a trained model | Data-built lists pick up odd words ("portuguese") |

---
# Part 6: Lesson connections

| Lesson | In the rule-based notebook |
|---|---|
| W1 s3–4: ML needs feature engineering, DL learns features | Rule-based is the far end of "hand-made": people decide the features (words) *and* their weights (scores) |
| W1 s5: neuron = weighted sum + activation | Our score is a weighted sum (word scores) → threshold. Structurally like a single perceptron (W1 s6) whose weights were set by people and formulas, not by backprop |
| W1 s6–8, W4 s7: perceptron limits (linear, XOR) | Why rules plateau: adding up word scores is linear, so context like "not bad" or "fast, but cold" needs extra rules |
| W1 s9–15: activations | VADER's squash `x/√(x²+15)` behaves like tanh |
| W1 s25, W4 s33: overfitting, validation | Building rules on train, choosing the cut-off on validation, testing once |
| W2 s5–8: NumPy, Pandas, Matplotlib | Counting words, tables, charts |
| W2 s14–16: venv, requirements | `sentiment_env`, `requirements.txt` (vaderSentiment) |
| W1 s24: types of learning | Rule-based isn't machine learning at all. Data was only used to build the word lists |

---
# Part 7: Questions the professor might ask

<details><summary><b>Q1. What is rule-based sentiment analysis?</b></summary>

Classifying text with a dictionary of words that have sentiment scores plus hand-written rules, like negation flipping a word or "but" giving more weight to what comes after it. No model is trained.
</details>

<details><summary><b>Q2. What are your results?</b></summary>

Plain VADER 79.6% on the test set, our improved rule-based system 88.6% (89.4% on validation).
</details>

<details><summary><b>Q3. Why VADER?</b></summary>

It's the most widely used rule-based sentiment tool, it's made for short informal text like reviews and social media, and its rules are well documented, so every score can be explained.
</details>

<details><summary><b>Q4. Why did plain VADER only get ~80%?</b></summary>

It was built for general text, and 16% of our reviews contained no word from its dictionary. Restaurant words like "waited", "slow", "cold", "fast" and "tasty" were missing.
</details>

<details><summary><b>Q5. How did you build the bigger dictionary?</b></summary>

For each word in the training reviews: 0.5 × log(share of positive reviews containing it ÷ share of negative reviews containing it). We kept words in at least 5 reviews that are at least 2.7× more common in one class, giving about 1,040 new words. This is called a corpus-based lexicon.
</details>

<details><summary><b>Q6. Isn't that machine learning?</b></summary>

The final system is a fixed list of scored words, phrases and patterns, and nothing is trained when scoring. Data was used to build the lists, the same way a person would read reviews to make a dictionary, just with a formula. It's related to how Naive Bayes scores words, so the line is a bit blurry, and we're open about that. The scores are combined through VADER's hand-written rules and our own phrase and pattern rules.
</details>

<details><summary><b>Q7. Why not use VADER's compound score anymore?</b></summary>

With over 1,000 extra words, most reviews have big totals, and the squash pushes them all near −1 or +1, so a mild review and a furious one look alike. Using the raw total (divided by √length) keeps that difference.
</details>

<details><summary><b>Q8. What was the negation bug?</b></summary>

VADER only treats "not" as a negation if "not" isn't in its dictionary. Our formula had given "not", "didn't" and others small scores, which switched off the rule. We found it by testing "not bad at all". We fixed it by scoring negation words separately, and accuracy went up.
</details>

<details><summary><b>Q9. Why add phrases?</b></summary>

VADER's dictionary only has single words, but meaning often comes from pairs: "not fresh", "not bad", "can't go wrong". A phrase only gets the extra meaning beyond its words, which can even be a correction in the opposite direction. "worst ever" gets positive points so the review isn't punished twice for "worst" and "ever".
</details>

<details><summary><b>Q9b. Why does "worst ever" have a positive phrase score?</b></summary>

The phrase score is only the extra meaning beyond the two words. "worst" and "ever" are each already strongly negative, so counting them plus a negative phrase score would punish the review twice. The positive correction balances it. The review as a whole still scores negative (\"Worst McDonald's ever\" gets -2.81, well below the cut-off).
</details>

<details><summary><b>Q10. Why is "bit slow" positive?</b></summary>

Because that's how people write: "a bit slow but the food was great" appears in good reviews, while angry reviews say "extremely slow" or "slowest ever". The data showed us something a person might not have guessed.
</details>

<details><summary><b>Q11. How did you choose the cut-off?</b></summary>

We tried 121 values on the validation set and kept the best (−0.254). The test set was only used once at the end.
</details>

<details><summary><b>Q12. Why not keep adding rules to reach 91%+?</b></summary>

We tried more (bigger dictionary, stronger scores, changing VADER's words, last-sentence weighting, a price rule). Only the star-rating and question rules helped, by about 0.4 points, and nothing passed ~89.5% on validation. Even ML models using the same word information stopped at ~90%. The remaining errors need context, sarcasm detection, or are label problems. Tuning further would only fit the validation set.
</details>

<details><summary><b>Q13. What are typical mistakes of your rule-based system?</b></summary>

Mixed reviews ("fast, but the fries were cold"), complaints with no negative words ("I asked for no pickles and got extra pickles"), sarcasm, and reviews whose stars don't match the text.
</details>

<details><summary><b>Q14. When would you use rule-based instead of the semantic model?</b></summary>

When there's no labeled data or GPU, when every decision must be explainable, or for a quick baseline. It runs instantly on any computer.
</details>

<details><summary><b>Q15. Why remove 3-star reviews?</b></summary>

They're usually mixed, so there's no reliable right answer. Public datasets built from star ratings do the same. Including them made every method drop to around 74–77%.
</details>

---
# Part 8: Self-check quiz

<details><summary><b>1. Range of VADER word scores and of the compound score?</b></summary>Words −4…+4. Compound −1…+1.</details>

<details><summary><b>2. Word scores sum to 3.0. Compound? (x / √(x² + 15))</b></summary>3 / √24 ≈ <b>0.612</b>.</details>

<details><summary><b>3. What does the "but" rule do?</b></summary>Words before "but" ×0.5, after ×1.5.</details>

<details><summary><b>4. "not" + "good" (1.9): what's the word's value after negation?</b></summary>1.9 × −0.74 ≈ <b>−1.41</b>.</details>

<details><summary><b>5. A word is in 2% of positive and 8% of negative training reviews. Its score?</b></summary>0.5 × log(0.02 / 0.08) = 0.5 × log(0.25) ≈ <b>−0.69</b>.</details>

<details><summary><b>6. Why divide by √(number of words)?</b></summary>So long reviews don't get extreme scores just because they have more words.</details>

<details><summary><b>7. Why must negation words stay out of VADER's dictionary?</b></summary>VADER only applies its negation rule using negation words that aren't in its dictionary.</details>

<details><summary><b>8. Which data was used to (a) build word lists, (b) choose the cut-off, (c) report the final score?</b></summary>(a) train, (b) validation, (c) test, once.</details>

<details><summary><b>9. Give one sentence a dictionary can't handle and why.</b></summary>"Hot coffee spilled on me". "hot" has one fixed (positive) score, and the dictionary can't use context.</details>

<details><summary><b>10. What is a corpus-based lexicon?</b></summary>A sentiment dictionary whose word scores are calculated from a collection of labeled texts instead of being assigned by people.</details>

<details><summary><b>11. Final test accuracies?</b></summary>Plain VADER 79.6%, hand-picked words 83.2%, final rule-based 88.6%.</details>